# QQQ下跌时SMH相对抗跌：同日5分钟来源与去重门

固定公式只用QQQ/SMH最后一根已完成5分钟raw NONE bar，QQQ该bar下跌时取SMH−QQQ对数收益幅度，否则0；在每个Qlib合法SOXL/SOXX 1分钟完成bar t向后匹配p≤t，p+1≤t+1决策。2018–2020来源发现，2021/22经济价值未读；与旧AMD/NVDA/SMH同涨比例、旧上一会话SMH−QQQ总收益和普通SOXX分钟收益分开。

In [1]:
import os,json,hashlib
from pathlib import Path
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';os.environ['MLFLOW_DISABLE_AGENT_HINT']='1'
import numpy as np
import pandas as pd
import qlib
from qlib.data import D
from src.core.recorder_sources import load_verified_recorder_objects
root=Path.cwd();card=root/'research/stage1_v5_peer_downshock_resilience_source_20261030';plan=json.loads((card/'plan.json').read_text());sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert sha(root/plan['peer_provider_uri']/'qlib_store_manifest.json')==plan['peer_provider_manifest_sha256'] and sha(root/plan['core_provider_uri']/'qlib_store_manifest.json')==plan['core_provider_manifest_sha256']
parent=root/plan['peer_parent_card'];assert sha(parent/'summary.json')==plan['peer_parent_summary_sha256'] and json.loads((parent/'nas_binding.json').read_text())['evidence']['sha256']==plan['peer_parent_nas_archive_sha256']
assert json.loads((parent/'summary.json').read_text())['source_id']==plan['peer_parent_source_id']
for old,digest in plan['closed_route_checks'].items():assert sha(root/old)==digest
label_card=root/plan['intent_source_card'];assert sha(label_card/'summary.json')==plan['intent_source_summary_sha256'] and sha(label_card/'verification_manifest.json')==plan['intent_source_manifest_sha256'] and json.loads((label_card/'nas_binding.json').read_text())['evidence']['sha256']==plan['intent_source_nas_archive_sha256']
qlib.init(provider_uri=str(root/plan['peer_provider_uri']),region='us',kernels=1)
peer=D.features(['QQQ','SMH'],['$open','$close','$price_valid'],start_time='2018-10-15 09:35:00',end_time='2020-12-31 16:00:00',freq='5min')
parts=[]
for symbol in ('QQQ','SMH'):
 x=peer.xs(symbol,level='instrument').copy();x.columns=[symbol+'_'+v.removeprefix('$') for v in x.columns];parts.append(x)
assert parts[0].index.equals(parts[1].index)
paired=parts[0].join(parts[1],how='inner').sort_index();assert paired.index.is_unique
paired['trade_date']=paired.index.normalize()
paired['qqq_ret']=np.log(paired.QQQ_close/paired.QQQ_open)
paired['smh_ret']=np.log(paired.SMH_close/paired.SMH_open)
paired['unconditional_gap']=paired.smh_ret-paired.qqq_ret
paired['downshock_resilience']=np.where(paired.qqq_ret<0,paired.unconditional_gap,0.)
paired['pair_valid']=paired[['QQQ_price_valid','SMH_price_valid']].eq(1).all(axis=1)
print('Qlib peer5m rows',len(peer),'paired',len(paired))

[47916:MainThread](2026-10-30 16:48:15,123) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[47916:MainThread](2026-10-30 16:48:15,607) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[47916:MainThread](2026-10-30 16:48:15,608) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/peer_5min')}


Qlib peer5m rows 86544 paired 43272


In [2]:
qlib.init(provider_uri=str(root/plan['core_provider_uri']),region='us',kernels=1)
core=D.features(['SOXX'],['$close','$price_valid'],start_time='2018-10-15 09:31:00',end_time='2020-12-31 16:00:00',freq='1min').xs('SOXX',level='instrument')
logclose=np.log(core['$close'].astype(float));ret5=logclose-logclose.groupby(core.index.normalize()).shift(5)
source={'card':plan['intent_source_card'],'recorder_id':plan['intent_source_recorder_id'],'manifest_sha256':plan['intent_source_manifest_sha256']}
v,_=load_verified_recorder_objects(root/source['card'],expected_recorder_id=source['recorder_id'],artifacts=['analysis/labels'],expected_manifest_sha256=source['manifest_sha256'])
labels=v['analysis/labels'];intents=labels.loc[labels.trade_date.dt.year.isin(plan['source_years']),['trade_date','bar_end_at','decision_at']].copy();assert intents.duplicated(['trade_date','bar_end_at']).sum()==0
intents=intents.sort_values('bar_end_at').reset_index(drop=True)
peer_rows=paired.loc[paired.pair_valid,['trade_date','downshock_resilience','unconditional_gap','qqq_ret','smh_ret']].reset_index().rename(columns={'datetime':'peer_bar_end_at'})
if 'peer_bar_end_at' not in peer_rows:peer_rows=peer_rows.rename(columns={peer_rows.columns[0]:'peer_bar_end_at'})
peer_rows=peer_rows.sort_values('peer_bar_end_at')
mapped=pd.merge_asof(intents,peer_rows,left_on='bar_end_at',right_on='peer_bar_end_at',by='trade_date',direction='backward',allow_exact_matches=True)
mapped['core_soxx_ret5']=ret5.reindex(pd.DatetimeIndex(mapped.bar_end_at)).to_numpy(float)
assert mapped.peer_bar_end_at.notna().all() and mapped.peer_bar_end_at.le(mapped.bar_end_at).all()
assert (mapped.peer_bar_end_at+pd.Timedelta(minutes=1)).le(mapped.decision_at).all()
assert mapped.downshock_resilience.notna().all()
mapped['year']=mapped.trade_date.dt.year
annual=[]
for year,g in mapped.groupby('year'):
 ranges=g.groupby('trade_date').downshock_resilience.agg(lambda x:float(x.max()-x.min()))
 annual.append({'year':int(year),'intent_rows':len(g),'source_days':int(g.trade_date.nunique()),'days_feature_range_gt_1bp':int(ranges.gt(.0001).sum()),'negative_qqq_bar_fraction':float(g.qqq_ret.lt(0).mean()),'feature_std':float(g.downshock_resilience.std())})
rank_gap=float(mapped.downshock_resilience.corr(mapped.unconditional_gap,method='spearman'))
rank_core=float(mapped.downshock_resilience.corr(mapped.core_soxx_ret5,method='spearman'))
gate=plan['gates'];byyear={x['year']:x for x in annual}
passed=bool(byyear[2018]['source_days']>=gate['minimum_2018_source_days'] and byyear[2019]['source_days']>=gate['minimum_2019_source_days'] and byyear[2020]['source_days']>=gate['minimum_2020_source_days'] and byyear[2018]['days_feature_range_gt_1bp']>=gate['minimum_2018_days_with_range_gt_1bp'] and byyear[2019]['days_feature_range_gt_1bp']>=gate['minimum_2019_days_with_range_gt_1bp'] and byyear[2020]['days_feature_range_gt_1bp']>=gate['minimum_2020_days_with_range_gt_1bp'] and abs(rank_gap)<gate['maximum_abs_discovery_spearman_with_unconditional_smh_qqq_gap'] and abs(rank_core)<gate['maximum_abs_discovery_spearman_with_core_soxx_ret5'])
audit={'experiment':plan['experiment'],'plan_sha256':sha(card/'plan.json'),'peer_qlib_rows':len(peer),'core_qlib_rows':len(core),'source_intent_rows':len(intents),'mapped_source_rows':int(mapped.peer_bar_end_at.notna().sum()),'maximum_peer_bar_age_minutes':float((mapped.bar_end_at-mapped.peer_bar_end_at).dt.total_seconds().max()/60),'annual':annual,'spearman_conditional_vs_unconditional_gap':rank_gap,'spearman_conditional_vs_core_soxx_ret5':rank_core,'source_gate_pass':passed,'historical_first_receipt_qualified':False,'future_value_columns_accessed':False,'new_model_fits':0,'new_candidate_accounts':0,'historical_research_passed':False,'decision':'same_day_qqq_downshock_smh_resilience_source_pass_needs_information' if passed else 'same_day_qqq_downshock_smh_resilience_source_failed'}
(card/'audit.json').write_text(json.dumps(audit,ensure_ascii=False,indent=2)+'\n')
pd.DataFrame(annual).to_csv(card/'annual_source_support.csv',index=False)
print(json.dumps(audit,ensure_ascii=False,indent=2))

[47916:MainThread](2026-10-30 16:48:15,842) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[47916:MainThread](2026-10-30 16:48:15,843) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[47916:MainThread](2026-10-30 16:48:15,843) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/stage1_event_guard_fullminute_20261028')}


{
  "experiment": "stage1_v5_peer_downshock_resilience_source_20261030",
  "plan_sha256": "0ad9b1e5ba2168cfbdaf1c2a5d45897e080f0c764f2569eb806b7b24d97dd6a4",
  "peer_qlib_rows": 86544,
  "core_qlib_rows": 216360,
  "source_intent_rows": 203526,
  "mapped_source_rows": 203526,
  "maximum_peer_bar_age_minutes": 4.0,
  "annual": [
    {
      "year": 2018,
      "intent_rows": 19091,
      "source_days": 53,
      "days_feature_range_gt_1bp": 53,
      "negative_qqq_bar_fraction": 0.5091928133675554,
      "feature_std": 0.0005841451929882169
    },
    {
      "year": 2019,
      "intent_rows": 91944,
      "source_days": 252,
      "days_feature_range_gt_1bp": 252,
      "negative_qqq_bar_fraction": 0.47478900200121815,
      "feature_std": 0.0004431285778991878
    },
    {
      "year": 2020,
      "intent_rows": 92491,
      "source_days": 253,
      "days_feature_range_gt_1bp": 253,
      "negative_qqq_bar_fraction": 0.47976559881501984,
      "feature_std": 0.0006056894781067967
  

/var/folders/4n/bc8v7c391nq228754jxg84_w0000gn/T/ipykernel_47916/358276114.py:14: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  assert (mapped.peer_bar_end_at+pd.Timedelta(minutes=1)).le(mapped.decision_at).all()


## 中文结论

SOXX历史Goal未通过。本卡零收益标签读取/拟合/账户。Qlib同行5min共86544行、SOXX核心1min共216360行；2018–2020固定原价股数标签的203526条决策前分钟意向全被QQQ/SMH最近已完成5min bar点时覆盖，同行bar最长滞后4分钟，按研究假定bar完成+1分钟≤接收器决策。2018/2019/2020的53/252/253天均有>1bp日内条件抗跌变化；负QQQ bar约47–51%。条件抗跌与无条件SMH−QQQ bar价差秩相关0.659、与普通SOXX ret5为0.184，预登记来源变化/去重门通过。旧AMD/NVDA/SMH三只同涨、旧前一完整会话SMH−QQQ累计gap作用域不同；本卡不重新拟合旧固定公式。2021/2022价值列未读、首次历史收取未知。下一步另冻同标签匹配信息门，控制含已失败的自身回撤、旧同行广度和无条件QQQ/SMH bar价差；源通过仍不代表可预测超额。